# Translate review-packet reports (local model)

Translation aid for the solo report audit (label-quality plan, step 4). It runs an
open-weight model on this Colab GPU: **no report text is sent to a hosted service.**

1. Upload the batch's `*_requests.jsonl` (from `outputs/label_quality/review/translate/`)
   to `MyDrive/knee_review/`.
2. Runtime → Change runtime type → **A100 GPU**. Run every cell top to bottom.
3. The self-check (cell 6) must pass before anything is translated. It uses synthetic
   sentences, not reports.
4. Download `*_translations.jsonl` from the same Drive folder and pass it to
   `build_blinded_packets.py render --translations`.

The translation is an aid only. Every ledger quote comes from the original report.

In [ ]:
import subprocess, torch
try:
    print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                         capture_output=True, text=True).stdout.strip())
except FileNotFoundError:
    print("nvidia-smi not found")
if not torch.cuda.is_available():
    raise SystemExit("No GPU. Runtime -> Change runtime type -> A100 GPU.")

In [ ]:
!pip install -q vllm
!pip uninstall -y -q torchaudio
import importlib.metadata
print("vllm", importlib.metadata.version("vllm"), "installed. Restart the runtime once if vllm fails to import below.")

In [ ]:
# Generated from src/labels/translate.py -- edit the module and regenerate, never here.
import importlib, pathlib, sys
TRANSLATE_SOURCE = '"""Translation aid for the report review: prompt, self-check and output checks.\n\nRuns inside `notebooks/22_translate_packets_colab.ipynb` on the participant\'s\nColab GPU with an open-weight model. No report text goes to a hosted service.\nThe notebook embeds this file verbatim, so the two can\'t drift apart.\n\nA translation is an aid, never evidence: the ledger\'s quote is always taken\nfrom the original report. The dangerous failure is a fluent translation that\ndrops a negation, a hedge or a side, so before any report is translated the\nmodel must pass `SELF_CHECKS`. These are synthetic sentences written for this\ncheck, not report text, and each one names the English terms a faithful\ntranslation must keep.\n"""\n\nimport hashlib\nimport re\n\nLANGUAGE_NAMES = {"el": "Greek", "de": "German", "tr": "Turkish", "es": "Spanish",\n                  "hr": "Croatian", "bg": "Bulgarian", "sr": "Serbian", "ru": "Russian",\n                  "it": "Italian", "fr": "French", "nl": "Dutch", "pt": "Portuguese",\n                  "cyrillic?": "a Cyrillic-script language", "unknown": "the source language"}\n\nSYSTEM = ("You translate knee MRI reports into English for a reviewer who will check every "\n          "statement against the original. Translate faithfully and completely. Keep every "\n          "negation (no, not, without, intact), every hedge (possible, suspected, probable, "\n          "cannot be excluded), every side and compartment (left, right, medial, lateral, "\n          "internal, external, anterior, posterior), every grade and number, and every "\n          "anatomical term. Do not add, omit, summarise, correct or interpret anything. "\n          "Output only the English translation.")\n\n# (language, synthetic sentence, groups of acceptable English terms; every group must match)\nSELF_CHECKS = [\n    ("de", "Kein Riss des lateralen Meniskus.",\n     [("no ", "without", "not "), ("lateral",), ("tear", "rupture")]),\n    ("es", "Posible rotura del menisco lateral.",\n     [("possible", "possibly", "suspected", "may "), ("lateral",), ("tear", "rupture")]),\n    ("el", "Ρήξη του έξω μηνίσκου.",\n     [("tear", "rupture"), ("lateral", "external", "outer")]),\n    ("tr", "Medial menisküs arka boynuzunda yırtık izlenmemiştir.",\n     [("no ", "not ", "without"), ("medial",), ("tear", "rupture"), ("posterior",)]),\n    ("hr", "Edem koštane srži lateralnog femoralnog kondila.",\n     [("marrow",), ("edema", "oedema"), ("lateral",), ("condyle",)]),\n    ("bg", "Хондромалация на пателата II степен.",\n     [("chondromalacia",), ("patella",), ("ii", "2", "second")]),\n    ("it", "Non si può escludere una contusione ossea.",\n     [("cannot be excluded", "cannot exclude", "can\'t be excluded", "can not be excluded",\n       "not be ruled out", "cannot be ruled out", "possible"), ("bone",),\n      ("contusion", "bruise")]),\n]\n\n\ndef messages(report, language):\n    source = LANGUAGE_NAMES.get(language, "the source language")\n    return [{"role": "system", "content": SYSTEM},\n            {"role": "user", "content": f"Translate this report from {source} into English.\\n\\n{report}"}]\n\n\ndef clean(text):\n    text = (text or "").strip()\n    return re.sub(r"^(english translation|translation)\\s*:\\s*", "", text, flags=re.I).strip()\n\n\ndef missing_terms(translation, groups):\n    """The groups a translation fails to cover; empty when it keeps everything."""\n    lowered = " " + clean(translation).lower() + " "\n    return [group for group in groups if not any(term in lowered for term in group)]\n\n\ndef self_check(translate_batch):\n    """Run the synthetic checks through `translate_batch(list of (text, language)) -> texts`.\n\n    Returns one failure per check that lost a required term, or an empty list.\n    """\n    outputs = translate_batch([(text, language) for language, text, _ in SELF_CHECKS])\n    failures = []\n    for (language, text, groups), output in zip(SELF_CHECKS, outputs):\n        lost = missing_terms(output, groups)\n        if lost:\n            failures.append({"language": language, "source": text, "translation": clean(output),\n                             "missing": [" / ".join(g).strip() for g in lost]})\n    return failures\n\n\ndef check_results(requests, results):\n    """One non-empty, untruncated translation per request, by case id.\n\n    `results` rows carry case_id, translation and finish_reason. Returns\n    problems by case id; never echoes text.\n    """\n    wanted = {r["case_id"] for r in requests}\n    got = {}\n    problems = []\n    for row in results:\n        if row["case_id"] in got:\n            problems.append(f"{row[\'case_id\']}: duplicate translation")\n        got[row["case_id"]] = row\n    for case_id in sorted(wanted - set(got)):\n        problems.append(f"{case_id}: missing translation")\n    for case_id in sorted(set(got) - wanted):\n        problems.append(f"{case_id}: translation for an unknown case")\n    for case_id in sorted(wanted & set(got)):\n        row = got[case_id]\n        if not clean(row.get("translation")):\n            problems.append(f"{case_id}: empty translation")\n        if row.get("finish_reason") == "length":\n            problems.append(f"{case_id}: truncated at the token limit")\n    return problems\n\n\ndef report_hash(report):\n    return hashlib.sha256(str(report).encode()).hexdigest()\n'
pathlib.Path("translate.py").write_text(TRANSLATE_SOURCE, encoding="utf-8")
sys.path.insert(0, ".")
importlib.invalidate_caches()
import translate

In [ ]:
import hashlib, json, pathlib
from google.colab import drive
drive.mount("/content/drive")
FOLDER = pathlib.Path("/content/drive/MyDrive/knee_review")
REQUESTS = sorted(FOLDER.glob("*_requests.jsonl"))
if len(REQUESTS) != 1:
    raise SystemExit(f"expected exactly one *_requests.jsonl in {FOLDER}, found {[p.name for p in REQUESTS]}")
REQUESTS = REQUESTS[0]
OUTPUT = FOLDER / REQUESTS.name.replace("_requests.jsonl", "_translations.jsonl")
requests = [json.loads(line) for line in REQUESTS.read_text(encoding="utf-8").splitlines() if line]
print(f"{len(requests)} reports to translate from {REQUESTS.name}")

In [ ]:
from vllm import LLM, SamplingParams
from transformers import AutoTokenizer

MODEL = "Qwen/Qwen2.5-14B-Instruct-AWQ"
MAX_TOKENS = 2048
tokenizer = AutoTokenizer.from_pretrained(MODEL)
llm = LLM(model=MODEL, dtype="auto", gpu_memory_utilization=0.90, max_model_len=8192,
          trust_remote_code=True)
SAMPLING = SamplingParams(temperature=0.0, max_tokens=MAX_TOKENS)


def generate(items):
    prompts = [tokenizer.apply_chat_template(translate.messages(text, language), tokenize=False,
                                             add_generation_prompt=True) for text, language in items]
    return [out.outputs[0] for out in llm.generate(prompts, SAMPLING)]


def translate_batch(items):
    return [o.text for o in generate(items)]

In [ ]:
# Self-check on synthetic sentences: every negation, hedge, side and grade must survive.
failures = translate.self_check(translate_batch)
for f in failures:
    print(f"FAIL {f['language']}: {f['source']!r} -> {f['translation']!r}; missing {f['missing']}")
if failures:
    raise SystemExit("The model lost required terms on synthetic checks; do not use its translations.")
print(f"self-check passed on {len(translate.SELF_CHECKS)} synthetic sentences")

In [ ]:
outputs = generate([(r["report"], r["language"]) for r in requests])
results = [{"case_id": r["case_id"], "translation": translate.clean(o.text), "finish_reason": o.finish_reason,
            "model": MODEL} for r, o in zip(requests, outputs)]
problems = translate.check_results(requests, results)
if problems:
    print("\n".join(problems))
    raise SystemExit("translation output failed its checks; nothing written")
with OUTPUT.open("w", encoding="utf-8") as handle:
    for row in results:
        handle.write(json.dumps(row, ensure_ascii=False) + "\n")
print(f"wrote {len(results)} translations to {OUTPUT.name}; input sha256 "
      f"{hashlib.sha256(REQUESTS.read_bytes()).hexdigest()[:16]}")